### Building out a cleaning script:

Testing out functions and approaches to clean formatting mistakes.

#### Imports


In [ ]:
from bookstats.config import FILTERED_DATA
import pandas as pd
import numpy as np
from bookstats.formatting import clean_headers

df = pd.read_csv(FILTERED_DATA)
clean_headers(df)

### Column names

In [ ]:
print(df.columns)

**TODO:** Rename '  num_pages'

-----

### Zero values in number of pages and rating

Choosing NaN for non-skewed aggregates while preserving the rows.

In [ ]:
df["num_pages"].replace(0, np.nan)

In [ ]:
(df["average_rating"] != 0) & (df["ratings_count"] == 0)

-----

### Whitespace in strings + formatting errors

Testing out helper function, final script will be inside a [formatting module](../bookstats/formatting.py)

**Regex searches:**
- 2+ spaces into one space
- ' : ' into ': ' (?)
- insert space into lowercaseUppercase (?)

(?) Can this potentially harm some unconventional book titles? Perhaps if so, book title is the least risky space for flaw, as it's easily readable or fixable by ISBN, while author name or publisher name becomes unsearchable with formatting errors.

Testing them separately:

In [ ]:
# Basic clean, all regex will move into this function once tested:
def clean_string(col: pd.Series) -> pd.Series:
    cleaned_col = col.str.strip()
    return cleaned_col
    
print(clean_string(pd.Series("Barnes  Noble Classics")))

In [ ]:
df.info()

# Totals for reference: 
total_pub_pre = df["publisher"].nunique()
total_title_pre = df["title"].nunique()
total_authors_pre = df["authors"].nunique()

### Multiple space characters:

Publishers:

In [ ]:
# Take the list of names, return only if it matches the regex to see what kind of results I get:

mask_pub_spaces = df["publisher"].str.contains(r"^.+ {2,}.+$", na=False)
df[mask_pub_spaces][["publisher"]].count()

In [ ]:
df[mask_pub_spaces][["publisher"]].value_counts()

**Overview table**

In [ ]:
mask_pub_spaces = df["publisher"].str.contains(r"^.+ {2,}.+$", na=False)
mask_authors_spaces = df["authors"].str.contains(r"^.+ {2,}.+$", na=False)
mask_title_spaces = df["title"].str.contains(r"^.+ {2,}.+$", na=False)

results = {
    "publisher": df[mask_pub_spaces]["publisher"].nunique(),
    "authors": df[mask_authors_spaces]["authors"].nunique(),
    "title": df[mask_title_spaces]["title"].nunique()
}

pd.Series(results)


Titles have unusually high number of matches, checking:

In [ ]:
faulty_titles = df[mask_title_spaces]

faulty_titles["title_fixed"] = faulty_titles["title"].str.replace(r" {2,}"," ",regex=True)
faulty_titles[["title", "title_fixed"]]


In [ ]:
faulty_authors = df[mask_authors_spaces]

faulty_authors["author_fixed"] = faulty_authors["authors"].str.replace(r" {2,}"," ",regex=True)
faulty_authors[["authors", "author_fixed"]]


Conclusion: Overall tendency looks correct, many titles hace double space in series.

### Examining before and after


In [ ]:
df["publisher_clean"] = df["publisher"].str.replace(r" {2,}"," ",regex=True)

df[["publisher", "publisher_clean"]].head(100)

**Merged publishers:**
This is how many would have been showing as separate entity in reading due to flawed formatting:

In [ ]:
total_pub_post = df["publisher_clean"].nunique()
changes = total_pub_pre - total_pub_post

print(changes)

### Fixing colon formatting

Confirm this appears and may cause duplicates:

In [ ]:
mask_colon = df["title"].str.contains(r"^.+ \: .+$", na=False)

df[mask_colon].count()

Confirmation:

In [ ]:
faulty = df[mask_colon]

faulty["title_fixed"] = faulty["title"].str.replace(r" : ", ": ", regex=True)

faulty[["title", "title_fixed"]]

In [ ]:
df["title_fixed"] = df["title"].str.replace(r" : ", ": ", regex=True)

df[["title", "title_fixed"]]

### Check for camelCase

This contains possibly some formatting errors but is pretty high risk.

In [ ]:
mask_camel = df["title"].str.contains(r"^.+[a-z]+[A-Z]+.+$", na=False)

df[mask_camel].head(10)

Conclusion: Near completely false positive in the sample -> Dropping this part of script, as I don't want to affect titles such as 'How to Buy Sell & Profit on eBay:' or 'The John McPhee Reader'

### Date format fix
Making sure publication date is in ISO for easier comparison, calculations etc.


In [ ]:
df["publication_date"].head()

Current format: M/D/YEAR. 
No zeros, no alphabetical sorting of strings.

In [ ]:
df["publication_date"].isna().value_counts()

In [ ]:
# Check is something fails to parse through datetime

parsed = pd.to_datetime(df["publication_date"], format="%m/%d/%Y", errors="coerce")
bad = parsed.isna() & df["publication_date"].notna()

print(df.loc[bad, "publication_date"])

One faulty date instance: 11/31/2000 -> will be NULL, rather than guessing what was meant by it.

In [ ]:
df["publication_date_formatted"] = parsed.dt.strftime("%Y-%m-%d")

print(df["publication_date_formatted"].head())

### Further Cleaning TODO

- Titles are sometimes wrapped in " (need to be distinguished from deliberate/legitimate ones)